In [1]:
# CELL 1 — imports and configuration
from pathlib import Path
import os
import re
import json
import hashlib
import time

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 80)

print("Imports ready | pandas:", pd.__version__, "| numpy:", np.__version__)

Imports ready | pandas: 3.1.1 | numpy: 2.4.6


In [2]:
# CELL 2 — locate workstream and product output paths
cwd = Path.cwd().resolve()
workstream_override = os.environ.get("OSD575_WORKSTREAM_ROOT")

if workstream_override:
    WORKSTREAM_ROOT = Path(workstream_override).expanduser().resolve()
else:
    WORKSTREAM_ROOT = next(
        (p for p in [cwd, *cwd.parents] if p.name.lower() == "ml-workstream-nasa"),
        None
    )
    if WORKSTREAM_ROOT is None:
        parent = next((p for p in [cwd, *cwd.parents] if (p / "ml-workstream-nasa").is_dir()), None)
        WORKSTREAM_ROOT = parent / "ml-workstream-nasa" if parent else None

if WORKSTREAM_ROOT is None or not WORKSTREAM_ROOT.is_dir():
    raise FileNotFoundError(
        "Could not locate ml-workstream-nasa. Open Jupyter from that folder or a child folder, "
        "or set OSD575_WORKSTREAM_ROOT to its absolute path."
    )

PROCESSED_DIR = WORKSTREAM_ROOT / "data" / "processed"
REPORTS_DIR = WORKSTREAM_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables"
CONFIGS_DIR = WORKSTREAM_ROOT / "configs"

for folder in [PROCESSED_DIR, REPORTS_DIR, TABLES_DIR, CONFIGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

INPUT_PATH = PROCESSED_DIR / "osd575_measurements_long_primary.csv"
if not INPUT_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {INPUT_PATH}. Run 03_nasa_osd575_biomarker_processing.ipynb first."
    )

print("Workstream:", WORKSTREAM_ROOT)
print("Input:", INPUT_PATH)

Workstream: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-nasa
Input: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-nasa\data\processed\osd575_measurements_long_primary.csv


In [3]:
# CELL 3 — load the primary normalized measurements and validate its contract
start_time = time.perf_counter()
measurements = pd.read_csv(INPUT_PATH, low_memory=False)

required = [
    "measurement_record_id", "study_id", "assay_family", "assay_platform",
    "analyte", "value", "raw_result_text", "unit", "subject_id", "sample_id",
    "timepoint", "source_version", "source_file", "source_table",
    "source_row_number", "source_measurement_column", "numeric_parse_status"
]
missing = [c for c in required if c not in measurements.columns]
if missing:
    raise ValueError(f"Primary processed table is missing required columns: {missing}")

measurements["value_numeric"] = pd.to_numeric(measurements["value"], errors="coerce")
measurements["subject_id"] = measurements["subject_id"].astype("string").str.strip()
measurements["timepoint"] = measurements["timepoint"].astype("string").str.strip()
measurements["assay_family"] = measurements["assay_family"].astype("string").str.strip()
measurements["analyte"] = measurements["analyte"].astype("string").str.strip()
measurements["unit"] = measurements["unit"].astype("string").str.strip()

finite_mask = np.isfinite(measurements["value_numeric"].to_numpy(dtype=float, na_value=np.nan))
usable = measurements.loc[
    measurements["numeric_parse_status"].astype(str).str.lower().eq("numeric") & finite_mask
].copy()

print(f"Loaded {len(measurements):,} primary source rows.")
print(f"Numeric finite observations available for product export: {len(usable):,}.")
print(f"Rows excluded from numeric export (retained in upstream all-source/primary file): {len(measurements) - len(usable):,}.")
display(measurements.head(8))

if usable.empty:
    raise RuntimeError("No finite numeric observations are available. Inspect Notebook 03 normalization and validation outputs.")

Loaded 15,813 primary source rows.
Numeric finite observations available for product export: 15,813.
Rows excluded from numeric export (retained in upstream all-source/primary file): 0.


,measurement_record_id,study_id,assay_family,assay_platform,analyte,value,raw_result_text,unit,subject_id,sample_id,timepoint,source_version,source_file,source_table,source_sheet,source_row_number,source_measurement_column,normalization_layout,numeric_parse_status,value_numeric
0,OSD575-M0000510,OSD-575,comprehensive_metabolic_panel,Quest Diagnostics CMP,albumin_value_gram_per_deciliter,4.90,4.90,<NA>,<NA>,C001_serum_L-3,<NA>,TRANSFORMED,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,NaN,2,albumin_value_gram_per_deciliter,wide,numeric,4.90
1,OSD575-M0000511,OSD-575,comprehensive_metabolic_panel,Quest Diagnostics CMP,albumin_to_globulin_ratio_value,2.00,2.00,<NA>,<NA>,C001_serum_L-3,<NA>,TRANSFORMED,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,NaN,2,albumin_to_globulin_ratio_value,wide,numeric,2.00
2,OSD575-M0000512,OSD-575,comprehensive_metabolic_panel,Quest Diagnostics CMP,total_bilirubin_value_milligram_per_deciliter,1.00,1.00,<NA>,<NA>,C001_serum_L-3,<NA>,TRANSFORMED,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,NaN,2,total_bilirubin_value_milligram_per_deciliter,wide,numeric,1.00
3,OSD575-M0000513,OSD-575,comprehensive_metabolic_panel,Quest Diagnostics CMP,bun_to_creatinine_ratio_value,15.00,15.00,<NA>,<NA>,C001_serum_L-3,<NA>,TRANSFORMED,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,NaN,2,bun_to_creatinine_ratio_value,wide,numeric,15.00
4,OSD575-M0000514,OSD-575,comprehensive_metabolic_panel,Quest Diagnostics CMP,calcium_value_milligram_per_deciliter,9.60,9.60,<NA>,<NA>,C001_serum_L-3,<NA>,TRANSFORMED,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,NaN,2,calcium_value_milligram_per_deciliter,wide,numeric,9.60
5,OSD575-M0000515,OSD-575,comprehensive_metabolic_panel,Quest Diagnostics CMP,carbon_dioxide_value_millimol_per_liter,26.00,26.00,<NA>,<NA>,C001_serum_L-3,<NA>,TRANSFORMED,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,NaN,2,carbon_dioxide_value_millimol_per_liter,wide,numeric,26.00
6,OSD575-M0000516,OSD-575,comprehensive_metabolic_panel,Quest Diagnostics CMP,chloride_value_millimol_per_liter,102.00,102.00,<NA>,<NA>,C001_serum_L-3,<NA>,TRANSFORMED,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,NaN,2,chloride_value_millimol_per_liter,wide,numeric,102.00
7,OSD575-M0000517,OSD-575,comprehensive_metabolic_panel,Quest Diagnostics CMP,creatinine_value_milligram_per_deciliter,1.37,1.37,<NA>,<NA>,C001_serum_L-3,<NA>,TRANSFORMED,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,NaN,2,creatinine_value_milligram_per_deciliter,wide,numeric,1.37


In [4]:
# CELL 4 — conservative assay-family to health-domain mapping
# The domain reflects the named assay family only; it is not a diagnosis.
def map_health_domain(family):
    value = str(family).lower()
    if "comprehensive_metabolic" in value:
        return "metabolic"
    if "cardiovascular" in value:
        return "cardiovascular"
    if "immune" in value or "alamar" in value:
        return "immune"
    return "unclassified"


def slug(value):
    text = re.sub(r"[^a-zA-Z0-9]+", "_", str(value).strip().lower()).strip("_")
    return text or "unknown"


def stable_id(parts, prefix="OSD575"):
    canonical = "|".join("" if pd.isna(x) else str(x) for x in parts)
    digest = hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:20]
    return f"{prefix}-{digest}"

usable["health_domain"] = usable["assay_family"].map(map_health_domain)
usable["metric_code"] = usable.apply(
    lambda r: "osd575." + slug(r["assay_family"]) + "." + slug(r["analyte"]) + "." + slug(r["unit"]),
    axis=1
)

print("Domain assignment counts:")
display(usable.groupby(["assay_family", "health_domain"], dropna=False).size().rename("records").reset_index())

Domain assignment counts:


,assay_family,health_domain,records
0,cardiovascular_multiplex,cardiovascular,504
1,comprehensive_metabolic_panel,metabolic,425
2,immune_alamar_multiplex,immune,10908
3,immune_eve_multiplex,immune,3976


In [5]:
# CELL 5 — construct product observation records with complete provenance
# timepoint_text is deliberately retained; no measured_at timestamp is synthesized.
product = pd.DataFrame({
    "observation_id": usable.apply(
        lambda r: stable_id([
            r.get("study_id"), r.get("measurement_record_id"), r.get("source_file"),
            r.get("source_table"), r.get("source_row_number")
        ]), axis=1
    ),
    "observation_type": "laboratory_biomarker",
    "data_origin": "retrospective_research",
    "source_system": "NASA Open Science Data Repository",
    "study_id": usable["study_id"].fillna("OSD-575"),
    "subject_id": usable["subject_id"],
    "sample_id": usable["sample_id"],
    "timepoint_text": usable["timepoint"],
    "measured_at_utc": pd.NA,
    "health_domain": usable["health_domain"],
    "assay_family": usable["assay_family"],
    "assay_platform": usable["assay_platform"],
    "metric_code": usable["metric_code"],
    "analyte": usable["analyte"],
    "value": usable["value_numeric"],
    "unit": usable["unit"],
    "raw_result_text": usable["raw_result_text"],
    "source_version": usable["source_version"],
    "source_file": usable["source_file"],
    "source_table": usable["source_table"],
    "source_sheet": usable["source_sheet"] if "source_sheet" in usable.columns else pd.NA,
    "source_row_number": usable["source_row_number"],
    "source_measurement_column": usable["source_measurement_column"],
    "upstream_measurement_record_id": usable["measurement_record_id"],
    "clinical_interpretation": pd.NA,
    "clinical_threshold_applied": False,
    "is_live_telemetry": False,
})

# A missing subject or timepoint must remain visible; don't silently drop it.
product["record_quality_status"] = np.select(
    [(product["subject_id"].isna() | product["subject_id"].astype("string").str.strip().eq("")).fillna(False).to_numpy(dtype=bool),
     (product["timepoint_text"].isna() | product["timepoint_text"].astype("string").str.strip().eq("")).fillna(False).to_numpy(dtype=bool),
     product["unit"].isna() | product["unit"].astype("string").str.strip().eq("")],
    ["missing_subject_id", "missing_timepoint", "missing_unit"],
    default="numeric_record_with_provenance"
)

product_path = PROCESSED_DIR / "osd575_product_observations.csv"
product.to_csv(product_path, index=False)
print("Product observation records:", len(product))
display(product.head(10))
print("Saved:", product_path)

Product observation records: 15813


,observation_id,observation_type,data_origin,source_system,study_id,subject_id,sample_id,timepoint_text,measured_at_utc,health_domain,assay_family,assay_platform,metric_code,analyte,value,unit,raw_result_text,source_version,source_file,source_table,source_sheet,source_row_number,source_measurement_column,upstream_measurement_record_id,clinical_interpretation,clinical_threshold_applied,is_live_telemetry,record_quality_status
0,OSD575-ed67a0ea788d6762d50c,laboratory_biomarker,retrospective_research,NASA Open Science Data Repository,OSD-575,<NA>,C001_serum_L-3,<NA>,<NA>,metabolic,comprehensive_metabolic_panel,Quest Diagnostics CMP,osd575.comprehensive_metabolic_panel.albumin_v...,albumin_value_gram_per_deciliter,4.90,<NA>,4.90,TRANSFORMED,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,NaN,2,albumin_value_gram_per_deciliter,OSD575-M0000510,<NA>,False,False,missing_subject_id
1,OSD575-2c81825d7aaae2feb09b,laboratory_biomarker,retrospective_research,NASA Open Science Data Repository,OSD-575,<NA>,C001_serum_L-3,<NA>,<NA>,metabolic,comprehensive_metabolic_panel,Quest Diagnostics CMP,osd575.comprehensive_metabolic_panel.albumin_t...,albumin_to_globulin_ratio_value,2.00,<NA>,2.00,TRANSFORMED,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,NaN,2,albumin_to_globulin_ratio_value,OSD575-M0000511,<NA>,False,False,missing_subject_id
2,OSD575-a3c439d6dbce6b8bc270,laboratory_biomarker,retrospective_research,NASA Open Science Data Repository,OSD-575,<NA>,C001_serum_L-3,<NA>,<NA>,metabolic,comprehensive_metabolic_panel,Quest Diagnostics CMP,osd575.comprehensive_metabolic_panel.total_bil...,total_bilirubin_value_milligram_per_deciliter,1.00,<NA>,1.00,TRANSFORMED,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,NaN,2,total_bilirubin_value_milligram_per_deciliter,OSD575-M0000512,<NA>,False,False,missing_subject_id
3,OSD575-3523d1dd17ffc31e6cdf,laboratory_biomarker,retrospective_research,NASA Open Science Data Repository,OSD-575,<NA>,C001_serum_L-3,<NA>,<NA>,metabolic,comprehensive_metabolic_panel,Quest Diagnostics CMP,osd575.comprehensive_metabolic_panel.bun_to_cr...,bun_to_creatinine_ratio_value,15.00,<NA>,15.00,TRANSFORMED,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,NaN,2,bun_to_creatinine_ratio_value,OSD575-M0000513,<NA>,False,False,missing_subject_id
4,OSD575-e6110962b615bdadb0d7,laboratory_biomarker,retrospective_research,NASA Open Science Data Repository,OSD-575,<NA>,C001_serum_L-3,<NA>,<NA>,metabolic,comprehensive_metabolic_panel,Quest Diagnostics CMP,osd575.comprehensive_metabolic_panel.calcium_v...,calcium_value_milligram_per_deciliter,9.60,<NA>,9.60,TRANSFORMED,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,NaN,2,calcium_value_milligram_per_deciliter,OSD575-M0000514,<NA>,False,False,missing_subject_id
5,OSD575-cda121149942858046ed,laboratory_biomarker,retrospective_research,NASA Open Science Data Repository,OSD-575,<NA>,C001_serum_L-3,<NA>,<NA>,metabolic,comprehensive_metabolic_panel,Quest Diagnostics CMP,osd575.comprehensive_metabolic_panel.carbon_di...,carbon_dioxide_value_millimol_per_liter,26.00,<NA>,26.00,TRANSFORMED,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,NaN,2,carbon_dioxide_value_millimol_per_liter,OSD575-M0000515,<NA>,False,False,missing_subject_id
6,OSD575-ccaef195c6f662db7930,laboratory_biomarker,retrospective_research,NASA Open Science Data Repository,OSD-575,<NA>,C001_serum_L-3,<NA>,<NA>,metabolic,comprehensive_metabolic_panel,Quest Diagnostics CMP,osd575.comprehensive_metabolic_panel.chloride_...,chloride_value_millimol_per_liter,102.00,<NA>,102.00,TRANSFORMED,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,NaN,2,chloride_value_millimol_per_liter,OSD575-M0

Saved: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-nasa\data\processed\osd575_product_observations.csv


In [6]:
# CELL 6 — export JSONL for application/API import
# Convert pandas/NumPy missing and scalar values into JSON-safe native values.
def json_safe(value):
    if value is None or value is pd.NA:
        return None
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return value.item() if hasattr(value, "item") else value

jsonl_path = PROCESSED_DIR / "osd575_product_observations.jsonl"
with jsonl_path.open("w", encoding="utf-8") as handle:
    for row in product.to_dict(orient="records"):
        handle.write(json.dumps({k: json_safe(v) for k, v in row.items()}, ensure_ascii=False) + "\n")

print("JSONL written:", jsonl_path)
print("First JSON record:")
with jsonl_path.open("r", encoding="utf-8") as handle:
    print(handle.readline()[:1800])

JSONL written: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-nasa\data\processed\osd575_product_observations.jsonl
First JSON record:
{"observation_id": "OSD575-ed67a0ea788d6762d50c", "observation_type": "laboratory_biomarker", "data_origin": "retrospective_research", "source_system": "NASA Open Science Data Repository", "study_id": "OSD-575", "subject_id": null, "sample_id": "C001_serum_L-3", "timepoint_text": null, "measured_at_utc": null, "health_domain": "metabolic", "assay_family": "comprehensive_metabolic_panel", "assay_platform": "Quest Diagnostics CMP", "metric_code": "osd575.comprehensive_metabolic_panel.albumin_value_gram_per_deciliter.na", "analyte": "albumin_value_gram_per_deciliter", "value": 4.9, "unit": null, "raw_result_text": 4.9, "source_version": "TRANSFORMED", "source_file": "LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANSFORMED.csv", "source_table": "LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANSFORMED", "source_sheet": null, "so

In [7]:
# CELL 7 — generate biomarker catalog
catalog = (
    product.groupby(
        ["metric_code", "health_domain", "assay_family", "assay_platform", "analyte", "unit"],
        dropna=False
    )
    .agg(
        observed_records=("observation_id", "count"),
        distinct_subjects=("subject_id", "nunique"),
        distinct_timepoints=("timepoint_text", "nunique"),
        source_versions=("source_version", lambda x: "; ".join(sorted(set(map(str, x.dropna()))))),
    )
    .reset_index()
)

catalog_path = PROCESSED_DIR / "osd575_biomarker_catalog.csv"
catalog.to_csv(catalog_path, index=False)
display(catalog.head(30))
print("Catalog entries:", len(catalog), "| saved:", catalog_path)

,metric_code,health_domain,assay_family,assay_platform,analyte,unit,observed_records,distinct_subjects,distinct_timepoints,source_versions
0,osd575.cardiovascular_multiplex.a2_macroglobul...,cardiovascular,cardiovascular_multiplex,Eve Technologies HDCVD9,a2_macroglobulin_concentration_nanogram_per_mi...,<NA>,28,0,0,TRANSFORMED
1,osd575.cardiovascular_multiplex.a2_macroglobul...,cardiovascular,cardiovascular_multiplex,Eve Technologies HDCVD9,a2_macroglobulin_percent,<NA>,28,0,0,TRANSFORMED
2,osd575.cardiovascular_multiplex.agp_concentrat...,cardiovascular,cardiovascular_multiplex,Eve Technologies HDCVD9,agp_concentration_nanogram_per_milliliter,<NA>,28,0,0,TRANSFORMED
3,osd575.cardiovascular_multiplex.agp_percent.na,cardiovascular,cardiovascular_multiplex,Eve Technologies HDCVD9,agp_percent,<NA>,28,0,0,TRANSFORMED
4,osd575.cardiovascular_multiplex.crp_concentrat...,cardiovascular,cardiovascular_multiplex,Eve Technologies HDCVD9,crp_concentration_picogram_per_milliliter,<NA>,28,0,0,TRANSFORMED
5,osd575.cardiovascular_multiplex.crp_percent.na,cardiovascular,cardiovascular_multiplex,Eve Technologies HDCVD9,crp_percent,<NA>,28,0,0,TRANSFORMED
6,osd575.cardiovascular_multiplex.fetuin_a36_con...,cardiovascular,cardiovascular_multiplex,Eve Technologies HDCVD9,fetuin_a36_concentration_nanogram_per_milliliter,<NA>,28,0,0,TRANSFORMED
7,osd575.cardiovascular_multiplex.fetuin_a36_per...,cardiovascular,cardiovascular_multiplex,Eve Technologies HDCVD9,fetuin_a36_percent,<NA>,28,0,0,TRANSFORMED
8,osd575.cardiovascular_multiplex.fibrinogen_con...,cardiovascular,cardiovascular_multiplex,Eve Technologies HDCVD9,fibrinogen_concentration_nanogram_per_milliliter,<NA>,28,0,0,TRANSFORMED
9,osd575.cardiovascular_multiplex.fibrinogen_per...,cardiovascular,cardiovascular_multiplex,Eve Technologies HDCVD9,fibrinogen_percent,<NA>,28,0,0,TRANSFORMED


Catalog entries: 580 | saved: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-nasa\data\processed\osd575_biomarker_catalog.csv


In [8]:
# CELL 8 — descriptive within-person pre-flight reference comparisons
# Only timepoint labels explicitly indicating L-/pre-flight/baseline are treated as pre-flight.
# Only R+/post-flight labels are treated as post-flight. Unknown labels stay unclassified.
def classify_timepoint(label):
    text = str(label).strip().lower()
    if re.search(r"(?:^|\b)l\s*[-−]\s*\d+", text) or "preflight" in text.replace("-", "") or "pre-flight" in text or "baseline" in text:
        return "pre_flight"
    if re.search(r"(?:^|\b)r\s*\+\s*\d+", text) or "postflight" in text.replace("-", "") or "post-flight" in text:
        return "post_flight"
    return "unclassified"

longitudinal = product.copy()
longitudinal["timepoint_phase"] = longitudinal["timepoint_text"].map(classify_timepoint)
longitudinal["value"] = pd.to_numeric(longitudinal["value"], errors="coerce")

# Collapse technical/repeated records only within the exact subject/analyte/unit/timepoint key.
# Source-level records remain unchanged in product_observations.csv and JSONL.
key_cols = ["subject_id", "health_domain", "assay_family", "analyte", "unit", "timepoint_text", "timepoint_phase"]
per_visit = (
    longitudinal.dropna(subset=["subject_id", "analyte", "value"])
    .groupby(key_cols, dropna=False)
    .agg(value_at_visit=("value", "mean"), replicate_count=("value", "count"))
    .reset_index()
)

baseline_rows = per_visit.loc[per_visit["timepoint_phase"] == "pre_flight"]
baseline = (
    baseline_rows.groupby(["subject_id", "health_domain", "assay_family", "analyte", "unit"], dropna=False)
    .agg(personal_preflight_baseline=("value_at_visit", "median"), preflight_timepoints=("timepoint_text", lambda x: "; ".join(sorted(set(map(str, x))))), preflight_visit_count=("value_at_visit", "count"))
    .reset_index()
)

post = per_visit.loc[per_visit["timepoint_phase"] == "post_flight"].copy()
change = post.merge(
    baseline,
    on=["subject_id", "health_domain", "assay_family", "analyte", "unit"],
    how="left",
    validate="many_to_one"
)
change["absolute_change_from_preflight"] = change["value_at_visit"] - change["personal_preflight_baseline"]
change["percent_change_from_preflight"] = np.where(
    change["personal_preflight_baseline"].abs() > 1e-12,
    100 * change["absolute_change_from_preflight"] / change["personal_preflight_baseline"],
    np.nan
)
change["comparison_status"] = np.where(
    change["personal_preflight_baseline"].notna(),
    "descriptive_comparison_available",
    "no_preflight_baseline_available"
)

change_path = PROCESSED_DIR / "osd575_personal_baseline_changes.csv"
change.to_csv(change_path, index=False)
print("Post-flight visit rows:", len(post))
print("Rows with a matched pre-flight reference:", int(change["personal_preflight_baseline"].notna().sum()))
print("Rows without a pre-flight reference:", int(change["personal_preflight_baseline"].isna().sum()))
display(change.head(25))
print("Saved:", change_path)


Post-flight visit rows: 0
Rows with a matched pre-flight reference: 0
Rows without a pre-flight reference: 0


,subject_id,health_domain,assay_family,analyte,unit,timepoint_text,timepoint_phase,value_at_visit,replicate_count,personal_preflight_baseline,preflight_timepoints,preflight_visit_count,absolute_change_from_preflight,percent_change_from_preflight,comparison_status


Saved: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-nasa\data\processed\osd575_personal_baseline_changes.csv


In [9]:
# CELL 9 — define a versioned product observation schema
schema = {
    "schema_name": "osd575_product_observation",
    "schema_version": "1.0.0",
    "description": "Retrospective research biomarker observation normalized from NASA OSD-575.",
    "required_fields": [
        "observation_id", "observation_type", "data_origin", "source_system", "study_id",
        "subject_id", "timepoint_text", "health_domain", "assay_family", "metric_code",
        "analyte", "value", "unit", "source_version", "source_file", "source_table",
        "upstream_measurement_record_id", "is_live_telemetry"
    ],
    "field_semantics": {
        "observation_id": "Stable identifier derived from study and upstream source-row provenance.",
        "subject_id": "Study pseudonymous crew identifier as present in source data.",
        "timepoint_text": "Exact source visit/timepoint label; not a calendar timestamp.",
        "measured_at_utc": "Null unless an explicit timestamp exists in the source; this notebook does not synthesize one.",
        "health_domain": "Conservative assay-family grouping: metabolic, immune, cardiovascular, or unclassified.",
        "value": "Numeric value parsed by Notebook 03; preserve unit alongside it.",
        "unit": "Original unit from source; not inferred.",
        "clinical_interpretation": "Always null in this export; no diagnosis or clinical interpretation is generated.",
        "clinical_threshold_applied": "Always false.",
        "is_live_telemetry": "Always false; these are retrospective research observations."
    },
    "source_study": "OSD-575",
    "source_selection": "Reads only osd575_measurements_long_primary.csv produced by Notebook 03.",
    "generated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
}
schema_path = CONFIGS_DIR / "osd575_observation_schema.json"
schema_path.write_text(json.dumps(schema, indent=2), encoding="utf-8")
print("Schema saved:", schema_path)

Schema saved: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-nasa\configs\osd575_observation_schema.json


In [10]:
# CELL 10 — write the application integration contract
contract = """# OSD-575 Product Integration Contract\n\n## Purpose\n\nThis export provides retrospective NASA OSD-575 biomarker observations for controlled import into the Astronaut Health Monitoring application. It is a research-data adapter, not a clinical decision engine.\n\n## Files\n\n- `data/processed/osd575_product_observations.csv`: flat observation table.\n- `data/processed/osd575_product_observations.jsonl`: one JSON object per observation.\n- `data/processed/osd575_biomarker_catalog.csv`: analyte/unit catalog.\n- `data/processed/osd575_personal_baseline_changes.csv`: descriptive post-flight differences from each crew member's available pre-flight measurements.\n- `configs/osd575_observation_schema.json`: versioned field contract.\n\n## Import rules\n\n1. Treat every record as `retrospective_research`, never as a live telemetry message.\n2. Keep the source `subject_id`, `sample_id`, and `timepoint_text`; do not convert relative visit labels into invented timestamps.\n3. Preserve numeric `value` and its original `unit` together. Never combine different analytes or units.\n4. Retain `observation_id` and `upstream_measurement_record_id` for idempotent imports and traceability.\n5. Retain source version, file, table, sheet, and row provenance.\n6. Do not surface `personal_baseline_changes` as a medical alert. They are descriptive comparisons only.\n7. Do not apply clinical reference ranges, diagnostic labels, risk scores, or treatment recommendations from this dataset.\n8. Display a clear label such as `NASA OSD-575 research reference data` anywhere these records appear in a demo.\n\n## Domain mapping\n\n- `comprehensive_metabolic_panel` -> `metabolic`\n- `immune_*` -> `immune`\n- `cardiovascular_*` -> `cardiovascular`\n- unrecognized families -> `unclassified`\n\nThe mapping is based on assay family metadata only. It does not infer the clinical meaning of an individual analyte.\n\n## Timestamp semantics\n\n`timepoint_text` contains the original visit label. `measured_at_utc` remains null because this adapter does not infer an exact collection timestamp from a relative flight-time label.\n\n## Limitations\n\nOSD-575 is a very small Inspiration4 cohort. These observations support data-pipeline development and descriptive longitudinal exploration; they do not validate a generalized astronaut-health model.\n"""
contract_path = REPORTS_DIR / "osd575_product_integration_contract.md"
contract_path.write_text(contract, encoding="utf-8")
print("Integration contract saved:", contract_path)

Integration contract saved: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-nasa\reports\osd575_product_integration_contract.md


In [11]:
# CELL 11 — export summary and integrity checks
checks = []
for path in [product_path, jsonl_path, catalog_path, change_path, schema_path, contract_path]:
    checks.append({
        "file": str(path.relative_to(WORKSTREAM_ROOT)),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
    })
checks_df = pd.DataFrame(checks)
display(checks_df)

# Confirm JSONL line count and uniqueness of exported observation IDs.
with jsonl_path.open("r", encoding="utf-8") as handle:
    jsonl_count = sum(1 for line in handle if line.strip())

summary = {
    "study_id": "OSD-575",
    "input_primary_records": int(len(measurements)),
    "numeric_product_observations": int(len(product)),
    "unique_subjects": int(product["subject_id"].nunique(dropna=True)),
    "unique_analytes": int(product["metric_code"].nunique(dropna=True)),
    "health_domains": sorted(product["health_domain"].dropna().astype(str).unique().tolist()),
    "jsonl_records": int(jsonl_count),
    "observations_missing_subject_id": int(product["subject_id"].isna().sum()),
    "observations_missing_timepoint": int(product["timepoint_text"].isna().sum()),
    "observations_missing_unit": int(product["unit"].isna().sum()),
    "clinical_thresholds_applied": False,
    "live_telemetry": False,
    "runtime_seconds": round(time.perf_counter() - start_time, 2),
}
summary_path = REPORTS_DIR / "osd575_product_integration_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))

if not checks_df["exists"].all():
    raise RuntimeError("One or more integration outputs are missing.")
if jsonl_count != len(product):
    raise RuntimeError(f"JSONL record count ({jsonl_count}) does not match CSV record count ({len(product)}).")
if product["observation_id"].duplicated().any():
    raise RuntimeError("Duplicate observation_id values found; inspect source provenance keys.")

print("\nOSD-575 product integration export completed.")
print("These are retrospective research observations, not live astronaut telemetry or clinical alerts.")

,file,exists,size_bytes
0,data\processed\osd575_product_observations.csv,True,7927882
1,data\processed\osd575_product_observations.jsonl,True,17383628
2,data\processed\osd575_biomarker_catalog.csv,True,94322
3,data\processed\osd575_personal_baseline_change...,True,264
4,configs\osd575_observation_schema.json,True,1711
5,reports\osd575_product_integration_contract.md,True,2338


,value
study_id,OSD-575
input_primary_records,15813
numeric_product_observations,15813
unique_subjects,0
unique_analytes,572
health_domains,"[cardiovascular, immune, metabolic]"
jsonl_records,15813
observations_missing_subject_id,15813
observations_missing_timepoint,15813
observations_missing_unit,15813



OSD-575 product integration export completed.
These are retrospective research observations, not live astronaut telemetry or clinical alerts.
